In [0]:


# ============================================================
# 0. CONFIG PATH
# ============================================================

dbutils.widgets.text("path", "")

config_path = dbutils.widgets.get("path")


# ============================================================
# 1. IMPORTS
# ============================================================

import json

import pyspark.sql.functions as F

from common_utils.logging import get_logger
from common_utils.generic_functions import safe_cast


# ============================================================
# 2. LOGGER
# ============================================================

logger = get_logger("dim_order_gold")


# ============================================================
# 3. READ CONFIG
# ============================================================

logger.info(
    "Reading Gold configuration from %s",
    config_path
)

with open(config_path, "r") as f:
    config = json.load(f)

logger.info(
    "Gold configuration loaded successfully"
)


# ============================================================
# 4. SOURCE / TARGET CONFIGURATION
# ============================================================

source_config = config["source"]
target_config = config["target"]

source_name = source_config["source_name"]

source_table = source_config["source_table"]

target_table = target_config["target_table"]


logger.info(
    "[%s] Source table: %s",
    source_name,
    source_table
)

logger.info(
    "[%s] Target table: %s",
    source_name,
    target_table
)


# ============================================================
# 5. READ SOURCE TABLE
# ============================================================

logger.info(
    "[%s] Reading sales_orders from %s",
    source_name,
    source_table
)

sales_orders_df = spark.read.table(
    source_table
)


logger.info(
    "[%s] sales_orders loaded successfully",
    source_name
)


# ============================================================
# 6. TRANSFORM ORDER
# ============================================================

def transform_order(df, config):

    logger.info(
        "[%s] Starting dim_order transformation",
        source_name
    )

    df = df.select(
        "order_number",
        "customer_id",
        "order_timestamp",
        "number_of_line_items"
    )


    # --------------------------------------------------------
    # Convert order timestamp
    # --------------------------------------------------------

    df = df.withColumn(
        "order_timestamp",
        F.to_timestamp(
            F.col("order_timestamp")
        )
    )


    # --------------------------------------------------------
    # Create date key
    # --------------------------------------------------------

    df = df.withColumn(
        "order_date_key",
        F.date_format(
            F.col("order_timestamp"),
            config["transform"]["order_date_key_format"]
        ).cast("int")
    )


    # --------------------------------------------------------
    # Cast final columns
    # --------------------------------------------------------

    df = safe_cast(
        df,
        config.get("cast_columns", {})
    )


    # --------------------------------------------------------
    # Select final columns
    # --------------------------------------------------------

    df = df.select(
        *config["select_columns"]
    )


    # --------------------------------------------------------
    # One row per order
    # --------------------------------------------------------

    df = df.dropDuplicates(
        ["order_number"]
    )


    logger.info(
        "[%s] dim_order transformation completed",
        source_name
    )

    return df


# ============================================================
# 7. CREATE DIM ORDER
# ============================================================

logger.info(
    "[%s] Starting dim_order creation",
    source_name
)

dim_order = transform_order(
    sales_orders_df,
    config
)


# ============================================================
# 8. VALIDATE DATA
# ============================================================

logger.info(
    "[%s] Validating dim_order",
    source_name
)

record_count = dim_order.count()

logger.info(
    "[%s] dim_order record count: %s",
    source_name,
    record_count
)

if record_count == 0:

    raise ValueError(
        "dim_order contains zero records"
    )


duplicate_count = (
    dim_order
    .groupBy("order_number")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


if duplicate_count > 0:

    raise ValueError(
        "Duplicate order_number detected in dim_order"
    )


# ============================================================
# 9. DISPLAY
# ============================================================

dim_order.show()


# ============================================================
# 10. CREATE GOLD SCHEMA
# ============================================================

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS
    retaildataplatform.gold
""")


# ============================================================
# 11. WRITE GOLD TABLE
# ============================================================

logger.info(
    "[%s] Writing dim_order to %s",
    source_name,
    target_table
)

(
    dim_order
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(target_table)
)


logger.info(
    "[%s] Gold table %s written successfully",
    source_name,
    target_table
)


# ============================================================
# 12. FINAL VALIDATION
# ============================================================

final_count = (
    spark.read
    .table(target_table)
    .count()
)

logger.info(
    "[%s] Final dim_order record count: %s",
    source_name,
    final_count
)

logger.info(
    "[%s] dim_order pipeline completed successfully",
    source_name
)